A notebook to process LLM outputs into familiar formats.

## Data Loading

In [2]:

from dataset_processing import process_llm_jsonl_results, CLIRENER_LABELS_V1, transform_to_ner_format
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
import wandb
from EXPERIMENTS.evaluate import run_nervaluate, log_to_wandb

/home/p0l3/miniforge3/envs/clirener_finetune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Based on [`run_campaign()`](EXPERIMENTS/evaluate_gold.py):

## Processing

In [ ]:
# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading
llm_predictions_dir = "RESULTS/LLM_PREDICTIONS/ner_results_gemma_4_31b_it.jsonl" 
#"RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
# "RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
# "RESULTS/LLM_PREDICTIONS/ner_results_gemini_3_pro_preview.jsonl"
llm_name = llm_predictions_dir.split("/")[-1].replace("ner_results_", "").replace(".jsonl", "")
raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
print(llm_name)

# WANDB initialization
WANDB_PROJECT = "CLIRENER_GOLD_SEEDS"

# B. Initialize WandB Run
run_name = f"eval_GOLD_{llm_name}_zs"

# Start a fresh run for this evaluation
run = wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    reinit=True, # Allow multiple runs in one script
    config={
        "model_type": "LLM_"+llm_name,
        "model_id": llm_name,
        "training_dataset": "None",
        "evaluation_dataset": GOLD_DATASET_ID, # Explicitly logging this
        "seed": -1,
        "evaluation_scope": "ALL_SPLITS_MERGED"
    }
)

# D. Transform Predictions
print("--- Transforming Predictions ---")
model_predictions_transformed, tag_to_id = transform_to_ner_format(raw_predictions, target_labels)
assert bio_label_list == list(tag_to_id.keys()), "Gold dataset tag order != prediction tag order!"

pred_lookup = {}
# Iterate through all predictions (including duplicates/failed attempts in the JSONL)
for row in model_predictions_transformed[0]:
    # print(row)
    text = row['text']
    tags = row['ner_tags']
    
    # Check if tags are valid (not None) and not empty.
    # If a sentence was attempted twice (once failed, once success), 
    # this logic ensures we only store the successful one.
    if tags is not None and len(tags) > 0:
        pred_lookup[text] = tags

true_ids = []
pred_ids = []
missing_count = 0
mismatch_count = 0

for row in gold_data["test"]:
    text_key = row['text']
    
    # 3. Match by exact text
    if text_key in pred_lookup:
        p_tags = pred_lookup[text_key]
        
        # Additional safety: Ensure lengths match (LLM might have truncated text)
        if len(row['ner_tags']) == len(p_tags):
            true_ids.append(row['ner_tags'])
            pred_ids.append(p_tags)
        else:
            mismatch_count += 1
    else:
        missing_count += 1

if missing_count > 0:
    print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
if mismatch_count > 0:
    print(f"Warning: {mismatch_count} rows were ignored due to token length mismatch.")
    
print(f"aligned {len(true_ids)} rows for evaluation.")



# E. Calculate Metrics
# We pass the dataset's BIO scheme for ID mapping, and the target labels for reporting
results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

# F. Log to WandB
log_to_wandb(results, results_by_tag)

print(f"SUCCESS: {run_name}")
# print(results)
wandb.finish()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
Error: File RESULTS/LLM_PREDICTIONS/ner_results_gemma_4_31b_it.jsonl not found.
gemma_4_31b_it
--- Transforming Predictions ---


IndexError: list index out of range

## Validation

In [ ]:
import json
import os
import re

def try_parse_json_response(raw_response):
    """
    Attempts to parse the raw LLM response as JSON.
    Handles potential markdown JSON blocks if present.
    """
    if not raw_response or not isinstance(raw_response, str):
        return False, None
    
    clean_resp = raw_response.strip()
    # Strip markdown code blocks if the LLM wrapped the JSON
    if clean_resp.startswith("```"):
        lines = clean_resp.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        clean_resp = "\n".join(lines).strip()
    
    try:
        parsed = json.loads(clean_resp)
        return True, parsed
    except Exception:
        return False, None

def extract_proposed_entities(parsed_json):
    """
    Extracts raw entity text strings proposed by the LLM from the parsed JSON structure.
    Tolerates variations in JSON key nomenclature.
    """
    proposed = []
    if isinstance(parsed_json, list):
        items = parsed_json
    elif isinstance(parsed_json, dict):
        items = []
        for val in parsed_json.values():
            if isinstance(val, list):
                items.extend(val)
    else:
        return proposed

    for item in items:
        if isinstance(item, dict):
            # Check common keys for entity text
            for key in ["entity_text", "entity", "text", "string", "name"]:
                if key in item and isinstance(item[key], str):
                    proposed.append(item[key])
                    break
    return proposed

# ALTERATION 1: Updated to exclusive index checks and added Unicode dashes
def is_word_aligned(text, start, end):
    """
    Determines if an exclusive character-level span [start, end] aligns with 
    word/punctuation boundaries in the source text.
    """
    if not text or start < 0 or end > len(text):
        return False
    
    # Whitespace, standard punctuation, and unicode dashes (en-dash, em-dash)
    boundary_chars = set(".,;:!?()[]{}'\"`-/\\ \t\n\r–—‘’“”")
    
    # Check left boundary (character before start)
    if start == 0:
        left_ok = True
    else:
        left_ok = text[start - 1] in boundary_chars
        
    # Check right boundary (character at end, since end is exclusive)
    if end == len(text):
        right_ok = True
    else:
        right_ok = text[end] in boundary_chars
        
    return left_ok and right_ok

def run_extraction_harness_diagnostics(file_path):
    """
    Analyzes the JSONL predictions to calculate JSON parse failures,
    string-match failures, and word-boundary alignment rates.
    """
    total_tasks = 0
    json_parse_failures = 0
    
    total_proposed_entities = 0
    string_match_failures = 0
    
    total_matched_spans = 0
    word_aligned_spans = 0

    if not os.path.exists(file_path):
        print(f"Error: File {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            total_tasks += 1
            task = json.loads(line)
            
            input_text = task.get("input_text", "")
            raw_response = task.get("raw_llm_response", "")
            processed_entities = task.get("processed_entities", [])
            
            # 1. Evaluate JSON Parse Failure
            success, parsed_json = try_parse_json_response(raw_response)
            if not success:
                json_parse_failures += 1
                continue  # Skip match analysis if JSON is unparseable
            
            # 2. Evaluate String-Match Failure
            proposed_entities = extract_proposed_entities(parsed_json)
            for ent_text in proposed_entities:
                total_proposed_entities += 1
                # If proposed string is not found in input_text (case-insensitive)
                if ent_text.lower() not in input_text.lower():
                    string_match_failures += 1
            
            # 3. Evaluate Word-Boundary Alignment
            for ent in processed_entities:
                if "span" in ent:
                    total_matched_spans += 1
                    start, end = ent["span"][0], ent["span"][1]
                    if is_word_aligned(input_text, start, end):
                        word_aligned_spans += 1
                    else:
                        print(input_text)
                        # ALTERATION 2: Switched to an f-string to prevent extra printing spaces
                        print(f"[{input_text[start:end]}]")

    # Compute final diagnostic rates
    parse_fail_rate = (json_parse_failures / total_tasks) if total_tasks else 0.0
    match_fail_rate = (string_match_failures / total_proposed_entities) if total_proposed_entities else 0.0
    boundary_alignment_rate = (word_aligned_spans / total_matched_spans) if total_matched_spans else 0.0

    print(f"\n================ EXTRACTION HARNESS DIAGNOSTICS ================")
    print(f"File Analyzed: {file_path.split('/')[-1]}")
    print(f"Total Sentences Evaluated: {total_tasks}")
    print(f"----------------------------------------------------------------")
    print(f"1. JSON Parse Failures:       {json_parse_failures}/{total_tasks} ({parse_fail_rate:.2%})")
    print(f"2. String-Match Failures:     {string_match_failures}/{total_proposed_entities} ({match_fail_rate:.2%})")
    print(f"3. Word-Aligned Spans:        {word_aligned_spans}/{total_matched_spans} ({boundary_alignment_rate:.2%})")
    print(f"================================================================")
    
    return {
        "parse_fail_rate": parse_fail_rate,
        "match_fail_rate": match_fail_rate,
        "boundary_alignment_rate": boundary_alignment_rate
    }

# Execute Diagnostics on your prediction files
llm_predictions_dir = "RESULTS/LLM_PREDICTIONS/"

for file in os.listdir(llm_predictions_dir):
    if file.endswith(".jsonl"):
        print(file)
        run_extraction_harness_diagnostics(llm_predictions_dir+file)
        print("\n\n")

In [22]:
import json
import re
import glob
import os
from dataset_processing import tokenize_text

def get_valid_token_boundaries(text):
    """
    Determines all valid character indices that represent the start 
    or end of a token in the sentence.
    """
    tokens = tokenize_text(text)
    valid_starts = set()
    valid_ends = set()
    
    current_offset = 0
    for token in tokens:
        # Find token start, searching from the current offset
        start = text.find(token, current_offset)
        end = start + len(token)
        
        valid_starts.add(start)
        valid_ends.add(end)
        
        current_offset = end
        
    return valid_starts, valid_ends

def main():
    # 1. Locate all LLM prediction files
    llm_files = glob.glob("RESULTS/LLM_PREDICTIONS/*.jsonl")
    
    if not llm_files:
        print("No LLM prediction files found in RESULTS/LLM_PREDICTIONS/")
        return

    print("--- LLM Harness Robustness Evaluation ---")
    
    overall_stats = {
        "total_prompts": 0,
        "json_parse_failures": 0,
        "total_extracted_entities": 0,
        "string_match_failures": 0,
        "matched_entities": 0,
        "mid_token_cuts": 0,
        "word_aligned": 0
    }

    for file_path in llm_files:
        model_name = os.path.basename(file_path).replace("ner_results_", "").replace(".jsonl", "")
        
        stats = {
            "total_prompts": 0,
            "json_parse_failures": 0,
            "total_extracted_entities": 0,
            "string_match_failures": 0,
            "matched_entities": 0,
            "mid_token_cuts": 0,
            "word_aligned": 0
        }

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                
                task = json.loads(line)
                
                stats["total_prompts"] += 1
                overall_stats["total_prompts"] += 1
                
                # --- CHECK 1: JSON Parse Failure ---
                # A parse failure is usually denoted by the absence of 'processed_entities'
                # or the explicit presence of an 'error' flag in the generation pipeline.
                has_error = "error" in task
                has_processed = "processed_entities" in task
                
                if has_error or not has_processed:
                    stats["json_parse_failures"] += 1
                    overall_stats["json_parse_failures"] += 1
                    continue # Skip boundary checks if it completely failed to parse
                    
                text = task.get("input_text", "")
                valid_starts, valid_ends = get_valid_token_boundaries(text)
                
                for ent in task.get("processed_entities", []):
                    stats["total_extracted_entities"] += 1
                    overall_stats["total_extracted_entities"] += 1
                    
                    # --- CHECK 2: String Match Failure ---
                    # The harness couldn't locate the predicted string in the source text
                    if "span" not in ent or ent["span"] is None or len(ent["span"]) != 2:
                        stats["string_match_failures"] += 1
                        overall_stats["string_match_failures"] += 1
                    else:
                        stats["matched_entities"] += 1
                        overall_stats["matched_entities"] += 1
                        
                        char_start, char_end = ent["span"]
                        
                        # --- CHECK 3: Boundary Alignment ---
                        # If the extracted character boundaries align with the tokenizer's 
                        # word boundaries, it's a linguistic choice, not a mechanical slice.
                        if char_start in valid_starts and char_end in valid_ends:
                            stats["word_aligned"] += 1
                            overall_stats["word_aligned"] += 1
                        else:
                            # print(text)
                            # print(ent)
                            stats["mid_token_cuts"] += 1
                            overall_stats["mid_token_cuts"] += 1

        # Print per-model stats
        parse_fail_pct = (stats["json_parse_failures"] / stats["total_prompts"] * 100) if stats["total_prompts"] > 0 else 0
        match_fail_pct = (stats["string_match_failures"] / stats["total_extracted_entities"] * 100) if stats["total_extracted_entities"] > 0 else 0
        word_align_pct = (stats["word_aligned"] / stats["matched_entities"] * 100) if stats["matched_entities"] > 0 else 0
        
        print(f"\nModel: {model_name}")
        print(f"  Parse Failures:  {stats['json_parse_failures']}/{stats['total_prompts']} ({parse_fail_pct:.2f}%)")
        print(f"  Match Failures:  {stats['string_match_failures']}/{stats['total_extracted_entities']} ({match_fail_pct:.2f}%)")
        print(f"  Word Aligned:    {stats['word_aligned']}/{stats['matched_entities']} ({word_align_pct:.2f}%)")
        print(f"  Mid-Token Cuts:  {stats['mid_token_cuts']}/{stats['matched_entities']} ({(100-word_align_pct):.2f}%)")

    # --- PRINT OVERALL STATS & PAPER DRAFT ---
    print("\n" + "="*60)
    print("OVERALL STATISTICS (For the Paper)")
    print("="*60)
    
    N = overall_stats["total_prompts"]
    X = (overall_stats["json_parse_failures"] / overall_stats["total_prompts"] * 100) if overall_stats["total_prompts"] > 0 else 0
    Y = (overall_stats["string_match_failures"] / overall_stats["total_extracted_entities"] * 100) if overall_stats["total_extracted_entities"] > 0 else 0
    Z = (overall_stats["word_aligned"] / overall_stats["matched_entities"] * 100) if overall_stats["matched_entities"] > 0 else 0
    
    print("\nDraft text for Section 6.2:")
    print("-" * 60)
    text_draft = (
        f"To rule out that observed boundary errors reflect artefacts of the string-matching extraction harness rather than genuine model behaviour, "
        f"we analysed {N} prompt responses and {overall_stats['total_extracted_entities']} extracted entities across all tested LLMs. "
        f"Only {X:.2f}% of outputs failed to parse as valid JSON, and {Y:.2f}% of extracted entity strings could not be located in the source sentence. "
        f"Crucially, among the successfully matched predictions, {Z:.2f}% align cleanly with word boundaries (suggesting a legitimate but incorrectly-scoped span, "
        f"e.g. omitting a modifier) rather than cutting mid-token. This strongly indicates that boundary errors predominantly reflect model-level scope decisions "
        f"rather than harness-induced corruption."
    )
    print(text_draft)
    print("-" * 60)

if __name__ == "__main__":
    main()

--- LLM Harness Robustness Evaluation ---

Model: gemini_3_pro_preview
  Parse Failures:  0/264 (0.00%)
  Match Failures:  0/2223 (0.00%)
  Word Aligned:    2220/2223 (99.87%)
  Mid-Token Cuts:  3/2223 (0.13%)

Model: gpt_5_2_pro
  Parse Failures:  0/230 (0.00%)
  Match Failures:  0/1740 (0.00%)
  Word Aligned:    1739/1740 (99.94%)
  Mid-Token Cuts:  1/1740 (0.06%)

Model: deepseek_chat
  Parse Failures:  0/192 (0.00%)
  Match Failures:  0/1534 (0.00%)
  Word Aligned:    1531/1534 (99.80%)
  Mid-Token Cuts:  3/1534 (0.20%)

Model: gemini_2_5_pro
  Parse Failures:  0/192 (0.00%)
  Match Failures:  0/2105 (0.00%)
  Word Aligned:    2103/2105 (99.90%)
  Mid-Token Cuts:  2/2105 (0.10%)

Model: deepseek_reasoner
  Parse Failures:  0/192 (0.00%)
  Match Failures:  0/1719 (0.00%)
  Word Aligned:    1717/1719 (99.88%)
  Mid-Token Cuts:  2/1719 (0.12%)

Model: claude_sonnet_4_5
  Parse Failures:  0/199 (0.00%)
  Match Failures:  0/1605 (0.00%)
  Word Aligned:    1602/1605 (99.81%)
  Mid-Token 

In [7]:
import os
# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading

# Execute Diagnostics on your prediction files
llm_predictions_DIR = "RESULTS/LLM_PREDICTIONS/"

for file in os.listdir(llm_predictions_DIR):
    if file.endswith(".jsonl"):
        llm_predictions_dir = llm_predictions_DIR + file 
        print(llm_predictions_dir)
        #"RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
        # "RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl"
        # "RESULTS/LLM_PREDICTIONS/ner_results_gemini_3_pro_preview.jsonl"
        llm_name = llm_predictions_dir.split("/")[-1].replace("ner_results_", "").replace(".jsonl", "")
        raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
        print(llm_name)

        # # WANDB initialization
        # WANDB_PROJECT = "CLIRENER_GOLD_SEEDS"

        # # B. Initialize WandB Run
        # run_name = f"eval_GOLD_{llm_name}_zs"

        # # Start a fresh run for this evaluation
        # run = wandb.init(
        #     project=WANDB_PROJECT,
        #     name=run_name,
        #     reinit=True, # Allow multiple runs in one script
        #     config={
        #         "model_type": "LLM_"+llm_name,
        #         "model_id": llm_name,
        #         "training_dataset": "None",
        #         "evaluation_dataset": GOLD_DATASET_ID, # Explicitly logging this
        #         "seed": -1,
        #         "evaluation_scope": "ALL_SPLITS_MERGED"
        #     }
        # )

        # D. Transform Predictions
        print("--- Transforming Predictions ---")
        model_predictions_transformed, tag_to_id = transform_to_ner_format(raw_predictions, target_labels)
        assert bio_label_list == list(tag_to_id.keys()), "Gold dataset tag order != prediction tag order!"

        pred_lookup = {}
        # Iterate through all predictions (including duplicates/failed attempts in the JSONL)
        # print(model_predictions_transformed)
        for row in model_predictions_transformed:
            # print(row)
            text = row['text']
            tags = row['ner_tags']
            
            # Check if tags are valid (not None) and not empty.
            # If a sentence was attempted twice (once failed, once success), 
            # this logic ensures we only store the successful one.
            if tags is not None and len(tags) > 0:
                pred_lookup[text] = tags

        true_ids = []
        pred_ids = []
        missing_count = 0
        mismatch_count = 0

        for row in gold_data["test"]:
            text_key = row['text']
            
            # 3. Match by exact text
            if text_key in pred_lookup:
                p_tags = pred_lookup[text_key]
                
                # Additional safety: Ensure lengths match (LLM might have truncated text)
                if len(row['ner_tags']) == len(p_tags):
                    true_ids.append(row['ner_tags'])
                    pred_ids.append(p_tags)
                else:
                    mismatch_count += 1
            else:
                missing_count += 1

        if missing_count > 0:
            print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
        if mismatch_count > 0:
            print(f"Warning: {mismatch_count} rows were ignored due to token length mismatch.")
            
        print(f"aligned {len(true_ids)} rows for evaluation.")



        # E. Calculate Metrics
        # We pass the dataset's BIO scheme for ID mapping, and the target labels for reporting
        results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

        # # F. Log to WandB
        # log_to_wandb(results, results_by_tag)

        # print(f"SUCCESS: {run_name}")
        # # print(results)
        # wandb.finish()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
RESULTS/LLM_PREDICTIONS/ner_results_gemini_3_pro_preview.jsonl
gemini_3_pro_preview
--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---
RESULTS/LLM_PREDICTIONS/ner_results_gpt_5_2_pro.jsonl
gpt_5_2_pro
--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---
RESULTS/LLM_PREDICTIONS/ner_results_deepseek_chat.jsonl
deepseek_chat
--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---
RESULTS/LLM_PREDICTIONS/ner_results_gemini_2_5_pro.jsonl
gemini_2_5_pro
--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---
RESULTS/LLM_PREDICTIONS/ner_results_deepseek_reasoner.jsonl
deepseek_reasoner
--- Transforming Predictions ---
aligned 192 rows for evaluation.
--- Calculating Metrics ---
RESULTS/LLM_PRED

## Boundary Missclassification Exmaples

In [6]:
import os
import random
from seqeval.metrics.sequence_labeling import get_entities

# Import your existing functions (adjust paths if needed based on where you run this)
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
from dataset_processing import process_llm_jsonl_results, transform_to_ner_format, CLIRENER_LABELS_V1
from EXPERIMENTS.evaluate import run_nervaluate

# True data loading
GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD"
gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
target_labels = list(CLIRENER_LABELS_V1)

# Predictions loading
llm_predictions_DIR = "RESULTS/LLM_PREDICTIONS/"

for file in os.listdir(llm_predictions_DIR):
    if file.endswith(".jsonl"):
        llm_predictions_dir = os.path.join(llm_predictions_DIR, file)
        llm_name = file.replace("ner_results_", "").replace(".jsonl", "")
        print(f"\n{'='*80}")
        print(f"Evaluating Model: {llm_name}")
        print(f"{'='*80}")
        
        raw_predictions = process_llm_jsonl_results(llm_predictions_dir)

        print("--- Transforming Predictions ---")
        model_predictions_transformed, tag_to_id = transform_to_ner_format(raw_predictions, target_labels)
        assert bio_label_list == list(tag_to_id.keys()), "Gold dataset tag order != prediction tag order!"

        pred_lookup = {}
        # Iterate through all predictions
        for row in model_predictions_transformed:
            text = row['text']
            tags = row['ner_tags']
            
            if tags is not None and len(tags) > 0:
                pred_lookup[text] = tags

        true_ids = []
        pred_ids = []
        aligned_tokens = [] # NEW: Keep track of tokens for printing examples
        missing_count = 0
        mismatch_count = 0

        for row in gold_data["test"]:
            text_key = row['text']
            
            # Match by exact text
            if text_key in pred_lookup:
                p_tags = pred_lookup[text_key]
                
                # Ensure lengths match
                if len(row['ner_tags']) == len(p_tags):
                    true_ids.append(row['ner_tags'])
                    pred_ids.append(p_tags)
                    aligned_tokens.append(row['tokens']) # Store tokens
                else:
                    mismatch_count += 1
            else:
                missing_count += 1

        if missing_count > 0:
            print(f"Warning: {missing_count} rows from Gold Data were not found in Predictions.")
        if mismatch_count > 0:
            print(f"Warning: {mismatch_count} rows were ignored due to token length mismatch.")
            
        print(f"Aligned {len(true_ids)} rows for evaluation.")

        # --- NEW: BOUNDARY ERROR EXTRACTION FOR APPENDIX ---
        boundary_errors = []
        
        # Iterate through our perfectly aligned lists
        for t_ids, p_ids, tokens in zip(true_ids, pred_ids, aligned_tokens):
            # Convert integer IDs back to string BIO tags
            t_tags = [bio_label_list[i] for i in t_ids]
            p_tags = [bio_label_list[i] for i in p_ids]
            
            # Use seqeval to cleanly extract (label, start_index, end_index)
            t_ents = get_entities(t_tags)
            p_ents = get_entities(p_tags)
            
            for p_lbl, p_start, p_end in p_ents:
                for t_lbl, t_start, t_end in t_ents:
                    # Check if the spans overlap at all (max of starts <= min of ends)
                    if max(p_start, t_start) <= min(p_end, t_end):
                        
                        # We specifically want cases where the model chose the CORRECT label,
                        # but got the boundary wrong (linguistic scope error)
                        if p_lbl == t_lbl and (p_start != t_start or p_end != t_end):
                            
                            gold_str = " ".join(tokens[t_start:t_end+1])
                            pred_str = " ".join(tokens[p_start:p_end+1])
                            
                            boundary_errors.append({
                                "sentence": " ".join(tokens),
                                "label": p_lbl,
                                "gold_str": gold_str,
                                "pred_str": pred_str,
                                "gold_span": f"[{t_start}:{t_end}]",
                                "pred_span": f"[{p_start}:{p_end}]"
                            })
                            
        # Print a random qualitative example if any boundary errors were found
        if boundary_errors:
            for i in range(3):
                example = random.choice(boundary_errors)
                print("\n" + "-"*60)
                print("📖 QUALITATIVE BOUNDARY ERROR EXAMPLE FOR APPENDIX:")
                print("-" * 60)
                print(f"Sentence:  {example['sentence']}")
                print(f"Entity:    {example['label']}")
                print(f"Gold:      \"{example['gold_str']}\" {example['gold_span']}")
                print(f"LLM Pred:  \"{example['pred_str']}\" {example['pred_span']}")
                print("-" * 60 + "\n")
        else:
            print("\nNo strict boundary errors found for this model.\n")

        # E. Calculate Metrics
        results, results_by_tag = run_nervaluate(true_ids, pred_ids, bio_label_list, tags=target_labels)

        # You can log to wandb here if you uncomment your wandb logic

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192

Evaluating Model: deepseek_chat
--- Transforming Predictions ---
Aligned 192 rows for evaluation.

------------------------------------------------------------
📖 QUALITATIVE BOUNDARY ERROR EXAMPLE FOR APPENDIX:
------------------------------------------------------------
Sentence:  From 2020 , there will be a requirement for all new buildings within the member states of the European Union to be ‘ nearly zero energy ’ ( European Union , 2010 ) : “ ‘ ’ … ” The Energy Performance in Buildings Directive ( EPBD ) places the responsibility for developing a legislative framework for the ‘ nearly zero-energy buildings ’ on the individual delivery of member states ; and are the UK ’ s response to this Directive .
Entity:    Policy
Gold:      "EPBD" [46:46]
LLM Pred:  "Energy Performance in Buildings Directive ( EPBD )" [40:47]
-------------------------

## Misclassification

In [7]:
import os
import json
from nervaluate import Evaluator

# Import existing loader/transformation functions from your codebase
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
from dataset_processing import process_llm_jsonl_results, transform_to_ner_format, CLIRENER_LABELS_V1

# Helper to convert integer IDs back to string labels for nervaluate
def ids_to_labels(pred_id_seqs, label_list):
    return [[label_list[i] for i in seq] for seq in pred_id_seqs]

def main():
    # 1. Load Gold Dataset
    GOLD_DATASET_ID = "P0L3/CliReNER_v_1_1_28_GOLD"
    gold_data, bio_label_list = load_and_merge_gold_data(GOLD_DATASET_ID)
    target_labels = list(CLIRENER_LABELS_V1)

    llm_predictions_DIR = "RESULTS/LLM_PREDICTIONS/"
    
    if not os.path.exists(llm_predictions_DIR):
        print(f"Directory {llm_predictions_DIR} not found.")
        return

    print("\n" + "="*80)
    print("🤖 RUNNING SCHEMA MISCLASSIFICATION ANALYSIS")
    print("="*80)

    for file in os.listdir(llm_predictions_DIR):
        if file.endswith(".jsonl"):
            llm_predictions_dir = os.path.join(llm_predictions_DIR, file)
            llm_name = file.replace("ner_results_", "").replace(".jsonl", "")
            
            # Replicate your exact alignment process
            raw_predictions = process_llm_jsonl_results(llm_predictions_dir)
            model_predictions_transformed, tag_to_id = transform_to_ner_format(raw_predictions, target_labels)
            
            pred_lookup = {}
            for row in model_predictions_transformed:
                text = row['text']
                tags = row['ner_tags']
                if tags is not None and len(tags) > 0:
                    pred_lookup[text] = tags

            true_ids = []
            pred_ids = []
            for row in gold_data["test"]:
                text_key = row['text']
                if text_key in pred_lookup:
                    p_tags = pred_lookup[text_key]
                    if len(row['ner_tags']) == len(p_tags):
                        true_ids.append(row['ner_tags'])
                        pred_ids.append(p_tags)

            if not true_ids:
                print(f"Skipping {llm_name}: No aligned rows.")
                continue

            # Convert aligned token-IDs back to string lists
            true_labels = ids_to_labels(true_ids, bio_label_list)
            pred_labels = ids_to_labels(pred_ids, bio_label_list)
            
            # 2. Run Nervaluate
            evaluator = Evaluator(true_labels, pred_labels, tags=target_labels, loader="list")
            results, _, _, _ = evaluator.evaluate()
            
            # 3. Extract the 'ent_type' metrics
            # In 'ent_type' mode, overlapping boundaries are grouped.
            # 'correct' means matching type; 'incorrect' means mismatched type.
            # 'missed' (FN) and 'spurious' (FP) denote zero-overlap detection failures.
            et = results['ent_type']
            correct = et.get('correct', 0)
            incorrect = et.get('incorrect', 0)
            missed = et.get('missed', 0)
            spurious = et.get('spurious', 0)
            
            total_errors = incorrect + missed + spurious
            confusion_rate = (incorrect / total_errors * 100) if total_errors > 0 else 0
            detection_rate = ((missed + spurious) / total_errors * 100) if total_errors > 0 else 0
            
            print(f"\nModel: {llm_name}")
            print(f"  - Correct Matches:            {correct}")
            print(f"  - Class Confusion Errors:   {incorrect} ({confusion_rate:.2f}% of total errors)")
            print(f"  - Detection Failures (FN+FP): {missed + spurious} ({detection_rate:.2f}% of total errors)")
            print(f"    └─ Completely Missed (FN):  {missed}")
            print(f"    └─ Spurious/Hallucinated:   {spurious}")

if __name__ == "__main__":
    main()

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192

🤖 RUNNING SCHEMA MISCLASSIFICATION ANALYSIS

Model: deepseek_chat
  - Correct Matches:            1207
  - Class Confusion Errors:   579 (45.88% of total errors)
  - Detection Failures (FN+FP): 683 (54.12% of total errors)
    └─ Completely Missed (FN):  662
    └─ Spurious/Hallucinated:   21

Model: claude_sonnet_4_5
  - Correct Matches:            1206
  - Class Confusion Errors:   511 (39.74% of total errors)
  - Detection Failures (FN+FP): 775 (60.26% of total errors)
    └─ Completely Missed (FN):  731
    └─ Spurious/Hallucinated:   44

Model: gemma_4_31b_it[1]
  - Correct Matches:            1436
  - Class Confusion Errors:   534 (61.81% of total errors)
  - Detection Failures (FN+FP): 330 (38.19% of total errors)
    └─ Completely Missed (FN):  284
    └─ Spurious/Hallucinated:   46

Model: gemma_4_31b_it
  - Correct Matches:          